In [ ]:
import pandas as pd
import numpy as np
import torch
import os

print("Loading raw data...")
log_train = pd.read_csv("data/raw/train/train/log_train.csv")
log_test = pd.read_csv("data/raw/test/test/log_test.csv")
object_df = pd.read_csv("data/raw/object.csv")
truth_train = pd.read_csv("data/raw/train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv("data/raw/test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

# ==========================================
# 1. FILTER FIRST 2 WEEKS
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

# ==========================================
# 2. MERGE CATEGORY
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)

# ==========================================
# 4. ENCODING (source / event / category streams)
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat

    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)

# ==========================================
# 5. VARIABLE-LENGTH SEQUENCES (no windowing, no padding)
# ==========================================
# Each student becomes exactly ONE sequence at its natural length.
# No MAX_LEN cap, no STRIDE, no truncation. Since lengths differ per
# student, we store each stream as a list of 1D tensors (ragged) rather
# than a single stacked array - padding is deferred to collate time in
# the training notebook, batch-by-batch.
def build_variable_length(df):
    src_list = [torch.LongTensor(s) for s in df["src"]]
    evt_list = [torch.LongTensor(e) for e in df["evt"]]
    cat_list = [torch.LongTensor(c) for c in df["cat"]]
    gap_list = [torch.FloatTensor(g) for g in df["gaps"]]
    y = torch.FloatTensor(df["dropout"].values)
    ids = df["student_id"].values
    return src_list, evt_list, cat_list, gap_list, y, ids

print("Building variable-length sequences (no windowing, no padding)...")
src_train, evt_train, cat_train, gap_train, y_train, _ = build_variable_length(train_df)
src_test, evt_test, cat_test, gap_test, y_test, test_ids = build_variable_length(test_df)

lengths_train = [len(s) for s in src_train]
print(f"Train sequence lengths -> min: {min(lengths_train)}, max: {max(lengths_train)}, mean: {np.mean(lengths_train):.1f}")

# ==========================================
# 6. SAVE PROCESSED DATA
# ==========================================
os.makedirs("data/processed", exist_ok=True)
torch.save({
    "train": (src_train, evt_train, cat_train, gap_train, y_train),
    "test": (src_test, evt_test, cat_test, gap_test, y_test, test_ids),
    "vocabs": (source_vocab, event_vocab, category_vocab),
    "truth_test": truth_test
}, "data/processed/processed_data.pt")

print("Success! Cleaned data saved to data/processed/processed_data.pt")